### Import relevant packages

In [29]:
from dotenv import load_dotenv
import os
import requests
import pandas as pd
import time

### Pull APIs and Extract Relevant Data

In [25]:
load_dotenv(dotenv_path="/Users/stella/Documents/DSProj/setlist-predictor/.env")
API_KEY = os.getenv("SETLISTFM_API_KEY")
HEADERS = {
    "x-api-key": API_KEY,
    "Accept": "application/json"
}

response = requests.get(
    "https://api.setlist.fm/rest/1.0/search/artists",
    headers=HEADERS,
    params={"artistName": "Taylor Swift"}
)
print(response.status_code)
data = response.json()
data

200


{'type': 'artists',
 'itemsPerPage': 30,
 'page': 1,
 'total': 104,
 'artist': [{'mbid': 'f44a46fb-18ea-4c2c-b578-0826b6e67631',
   'name': '3LAU vs. Calvin Harris vs. Taylor Swift',
   'sortName': '3LAU vs. Harris, Calvin vs. Swift, Taylor',
   'disambiguation': '',
   'url': 'https://www.setlist.fm/setlists/3lau-vs-calvin-harris-vs-taylor-swift-53c8e7a5.html'},
  {'mbid': '7288729a-b09e-4b5d-a9d6-a3ca8c42bd18',
   'name': 'Gracie Abrams feat. Taylor Swift',
   'sortName': 'Abrams, Gracie feat. Swift, Taylor',
   'disambiguation': '',
   'url': 'https://www.setlist.fm/setlists/gracie-abrams-feat-taylor-swift-4be51bde.html'},
  {'mbid': 'a26ca924-c7ba-4ab3-92e0-929145382fca',
   'name': 'Almost Eras: The Taylor Swift Experience',
   'sortName': 'Almost Eras: The Taylor Swift Experience',
   'disambiguation': 'Taylor Swift tribute band',
   'url': 'https://www.setlist.fm/setlists/almost-eras-the-taylor-swift-experience-2b9118ca.html'},
  {'mbid': '66fb2961-e3eb-42f4-ae64-640148d71cad',


In [26]:
def find_artist_mbid(artist_name, headers, exact_match=True):
    page = 1
    while True:
        response = requests.get(
            "https://api.setlist.fm/rest/1.0/search/artists",
            headers=headers,
            params={"artistName": artist_name, "p": page}
        )
        
        if response.status_code != 200:
            print(f"Stopped — status {response.status_code} on page {page}")
            print(response.json())
            break
        
        data = response.json()
        
        if "total" not in data:
            print(f"Unexpected response shape on page {page}:")
            print(data)
            break
        
        for artist in data.get("artist", []):
            if exact_match and artist["name"].lower() == artist_name.lower():
                return artist["mbid"], artist["name"]
            elif not exact_match and artist_name.lower() in artist["name"].lower():
                print(artist["name"], "-", artist["mbid"])
        
        total_pages = -(-data["total"] // data["itemsPerPage"])
        if page >= total_pages:
            break
        page += 1
        
        import time
        time.sleep(0.5)
    
    return None, None

In [27]:
mbid, name = find_artist_mbid("Taylor Swift", HEADERS)
print(mbid, name)

20244d07-534f-4eff-b4d4-930878889970 Taylor Swift


In [30]:
mbid = "20244d07-534f-4eff-b4d4-930878889970"
setlists = []
page = 1

while True:
    response = requests.get(
        f"https://api.setlist.fm/rest/1.0/artist/{mbid}/setlists",
        headers=HEADERS,
        params={"p": page}
    )
    if response.status_code != 200:
        print(f"Stopped at page {page}, status {response.status_code}")
        break
    
    page_data = response.json()
    setlists.extend(page_data.get("setlist", []))
    
    total_pages = -(-page_data["total"] // page_data["itemsPerPage"])
    print(f"Pulled page {page}/{total_pages}")
    
    if page >= total_pages:
        break
    page += 1
    
    time.sleep(1.5)

Pulled page 1/58
Pulled page 2/58
Pulled page 3/58
Pulled page 4/58
Pulled page 5/58
Pulled page 6/58
Pulled page 7/58
Pulled page 8/58
Pulled page 9/58
Pulled page 10/58
Pulled page 11/58
Pulled page 12/58
Pulled page 13/58
Pulled page 14/58
Pulled page 15/58
Pulled page 16/58
Pulled page 17/58
Pulled page 18/58
Pulled page 19/58
Pulled page 20/58
Pulled page 21/58
Pulled page 22/58
Pulled page 23/58
Pulled page 24/58
Pulled page 25/58
Pulled page 26/58
Pulled page 27/58
Pulled page 28/58
Pulled page 29/58
Pulled page 30/58
Pulled page 31/58
Pulled page 32/58
Pulled page 33/58
Pulled page 34/58
Pulled page 35/58
Pulled page 36/58
Pulled page 37/58
Pulled page 38/58
Pulled page 39/58
Pulled page 40/58
Pulled page 41/58
Pulled page 42/58
Pulled page 43/58
Pulled page 44/58
Pulled page 45/58
Pulled page 46/58
Pulled page 47/58
Pulled page 48/58
Pulled page 49/58
Pulled page 50/58
Pulled page 51/58
Pulled page 52/58
Pulled page 53/58
Pulled page 54/58
Pulled page 55/58
Pulled page 56/58
P

In [31]:
print(len(setlists)) 

1148


In [32]:
rows = []
for show in setlists:
    if not show["sets"]["set"]:
        continue
    
    for set_block in show["sets"]["set"]:
        set_name = set_block.get("name", "main")
        for song in set_block.get("song", []):
            rows.append({
                "event_date": show["eventDate"],
                "venue": show["venue"]["name"],
                "city": show["venue"]["city"]["name"],
                "country": show["venue"]["city"]["country"]["name"],
                "tour": show.get("tour", {}).get("name", None),
                "set_name": set_name,
                "song_name": song.get("name"),
                "is_cover": "cover" in song,
                "info": song.get("info", None)
            })

df = pd.DataFrame(rows)
df["event_date"] = pd.to_datetime(df["event_date"], format="%d-%m-%Y")
df.head()

,event_date,venue,city,country,tour,set_name,song_name,is_cover,info
0,2026-06-09,Dolby Theatre,Los Angeles,United States,NaN,main,"I Knew It, I Knew You",False,live debut; on piano
1,2026-06-09,Dolby Theatre,Los Angeles,United States,NaN,main,You've Got a Friend in Me,True,Taylor introduced Randy Newman
2,2024-12-08,BC Place Stadium,Vancouver,Canada,The Eras Tour,main,,False,"w/ elements of MA&tHP, The Alchemy, Fearless, ..."
3,2024-12-08,BC Place Stadium,Vancouver,Canada,The Eras Tour,Lover,Miss Americana & the Heartbreak Prince,False,shortened
4,2024-12-08,BC Place Stadium,Vancouver,Canada,The Eras Tour,Lover,Cruel Summer,False,extended outro


In [34]:
df.to_csv("../data/taylor_swift_setlists.csv", index=False)